# Export Training AP X-rays

Extracts all training AP images from GCS tarball, converts to lower-res PNGs, zips for download.

Output: `<DIR_NAME>_ap.png` for each training sample.

In [ ]:
GCS_DATA_URL = ""  # gs://your-bucket/training_data.tar.gz
OUTPUT_RES = 128    # output PNG resolution (original is 224x224)

assert GCS_DATA_URL, "Set GCS_DATA_URL above!"

In [ ]:
from google.colab import auth
auth.authenticate_user()
print("Authenticated")

In [ ]:
import os, json, subprocess

TARBALL = "/content/training_data.tar.gz"
DATA_DIR = "/content/training_data"
OUT_DIR = "/content/ap_images"
os.makedirs(OUT_DIR, exist_ok=True)

# Download tarball
if not os.path.exists(TARBALL):
    !gcloud storage cp "{GCS_DATA_URL}" "{TARBALL}"
print(f"Tarball: {os.path.getsize(TARBALL) / 1e9:.1f} GB")

In [ ]:
# Extract manifest to get training sample IDs
manifest_match = subprocess.run(
    ["tar", "tzf", TARBALL, "--wildcards", "*/manifest.json"],
    capture_output=True, text=True
)
manifest_tar_path = manifest_match.stdout.strip().split("\n")[0]
!tar xzf {TARBALL} -C /content/ "{manifest_tar_path}"

manifest = json.loads(open(f"/content/{manifest_tar_path}").read())
train_ids = manifest["train"]
tar_prefix = manifest_tar_path.replace("manifest.json", "")

print(f"Training samples: {len(train_ids)}")

In [ ]:
# Extract only ap.npy files for training samples
patterns_file = "/content/ap_patterns.txt"
with open(patterns_file, "w") as f:
    for sid in train_ids:
        f.write(f"{tar_prefix}{sid}/ap.npy\n")

print(f"Extracting {len(train_ids)} ap.npy files...")
!tar xzf {TARBALL} -C /content/ -T {patterns_file}

# Delete tarball immediately
os.remove(TARBALL)
os.remove(patterns_file)
print("Tarball deleted.")
!df -h /content | tail -1

In [ ]:
# Convert ap.npy -> lower-res PNG
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm

converted = 0
skipped = 0

for sid in tqdm(train_ids, desc="Converting"):
    npy_path = f"{DATA_DIR}/{sid}/ap.npy"
    if not os.path.exists(npy_path):
        skipped += 1
        continue

    arr = np.load(npy_path).astype(np.float32)
    img = Image.fromarray((arr * 255).clip(0, 255).astype(np.uint8))
    if OUTPUT_RES != img.size[0]:
        img = img.resize((OUTPUT_RES, OUTPUT_RES), Image.LANCZOS)
    img.save(f"{OUT_DIR}/{sid}_ap.png")
    converted += 1

print(f"\nConverted: {converted}, Skipped: {skipped}")

In [ ]:
# Clean up extracted npy files
import shutil
shutil.rmtree(DATA_DIR, ignore_errors=True)
print("Cleaned up npy files")
!df -h /content | tail -1

In [ ]:
# Zip and download
print("Zipping...")
shutil.make_archive("/content/ap_images", "zip", OUT_DIR)
zip_size = os.path.getsize("/content/ap_images.zip") / 1e6
print(f"ap_images.zip: {zip_size:.1f} MB ({converted} images)")

from google.colab import files
files.download("/content/ap_images.zip")